In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [22]:
!pip install -q faiss-cpu
!pip uninstall -y torchao -q

!pip install -q \
transformers==4.56.1 \
peft==0.17.1 \
accelerate==1.10.1 \
sentencepiece \
wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 93.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 29.6 MB/s eta 0:00:00


In [7]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [9]:
import os
import wandb

wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"

run = wandb.init(
    project="23f2000775-t22026",
    name="milestone3-rag",
    mode=wandb_mode,
    config={
    "embedding_model": embedding_model_name,
    "llm": llm_name,
    "top_k": top_k,
    "sample_size": sample_size
},
    reinit=True
)

wandb.log({
    "MAP@3": rag_map3,
    "Accuracy": rag_accuracy
})

run.finish()

NameError: name 'embedding_model_name' is not defined

In [ ]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(train_path)

print(train.shape)
train.head()

In [ ]:
train.isnull().sum()

In [ ]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [ ]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

In [ ]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

In [ ]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [ ]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [ ]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    max_features=50000,
    sublinear_tf=True
)

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

In [ ]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [ ]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

In [ ]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

In [ ]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

In [ ]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [ ]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

In [24]:
# %% [markdown]
# ## Leaderboard model — DeBERTa-v3 + LoRA multiple-choice classifier
#
# Paste these cells *after* the existing notebook cells.  This deliberately
# leaves the TF-IDF/Word2Vec work intact and does not use generation or RAG.

# %%
# Install only the packages required by the transformer block.  Restart the
# notebook kernel once if Kaggle upgrades a package during this cell.
!pip install -q -U "transformers>=4.45,<5" "peft>=0.13" "accelerate>=0.34" "sentencepiece>=0.2" "wandb>=0.18"

# %%
import gc
import glob
import os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForMultipleChoice,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Kaggle settings: switch to deberta-v3-large only after confirming it fits.
# v3-base is the default because it allows a larger effective batch on a T4.
CFG = {
    "model_name": "microsoft/deberta-v3-base",  # optional: microsoft/deberta-v3-large
    "max_length": 256,
    "learning_rate": 2e-5,
    "train_batch_size": 8,       # large: use 2 or 4
    "eval_batch_size": 16,       # large: use 4 or 8
    "gradient_accumulation": 2,  # effective batch = batch_size * this value
    "epochs": 4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.08,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "label_smoothing": 0.0,
    "validation_size": 0.12,
    "seed": 42,
    "retrain_on_full_train": True,
}

# A compact, deliberately conservative search grid.  For a deadline run,
# retain the defaults above; use this grid for a second notebook version only.
# SEARCH_SPACE = {
#     "learning_rate": [1e-5, 1.5e-5, 2e-5],
#     "train_batch_size": [4, 8],
#     "epochs": [3, 4],
#     "lora_r": [8, 16, 32],
#     "weight_decay": [0.0, 0.01],
#     "warmup_ratio": [0.05, 0.08, 0.10],
# }

set_seed(CFG["seed"])
torch.backends.cuda.matmul.allow_tf32 = False
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."
print(torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

# %%
# Locate competition files without assuming whether Kaggle mounted them under
# /competitions or directly under /kaggle/input/<competition>.
def locate_competition_file(filename):
    candidates = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} below /kaggle/input")
    preferred = [p for p in candidates if "smart-mcq-solver-challenge" in p]
    return preferred[0] if preferred else candidates[0]

train_path = locate_competition_file("train.csv")
test_path = locate_competition_file("test.csv")
sample_paths = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)

# Re-read raw text: transformer tokenization should preserve punctuation and
# casing, unlike the previous baseline's cleaned columns.
mcq_train = pd.read_csv(train_path)
mcq_test = pd.read_csv(test_path)
OPTION_LABELS = [c for c in ["A", "B", "C", "D", "E"] if c in mcq_train.columns]
assert len(OPTION_LABELS) >= 2, f"Expected option columns, got {mcq_train.columns.tolist()}"
assert "answer" in mcq_train.columns, "The training CSV must include answer."
assert set(mcq_train.answer.unique()).issubset(set(OPTION_LABELS))

for frame in (mcq_train, mcq_test):
    for col in ["prompt", *OPTION_LABELS]:
        frame[col] = frame[col].fillna("").astype(str)

label_to_id = {label: i for i, label in enumerate(OPTION_LABELS)}
mcq_train["labels"] = mcq_train["answer"].map(label_to_id).astype("int64")
print("Train/test:", mcq_train.shape, mcq_test.shape, "choices:", OPTION_LABELS)
display(mcq_train[["prompt", *OPTION_LABELS, "answer"]].head(2))

# %%
tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"]
)

def tokenize_mcq(batch, with_labels=True):
    num_rows = len(batch["prompt"])

    first_sentences = [[q] * len(OPTION_LABELS) for q in batch["prompt"]]
    second_sentences = [
        [f"{label}. {batch[label][i]}" for label in OPTION_LABELS]
        for i in range(num_rows)
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=CFG["max_length"],
    )

    tokenized = {
        k: [
            v[i:i + len(OPTION_LABELS)]
            for i in range(0, len(v), len(OPTION_LABELS))
        ]
        for k, v in tokenized.items()
    }

    if with_labels:
        tokenized["labels"] = batch["labels"]

    return tokenized

@dataclass
class MultipleChoiceCollator:
    tokenizer: object

    def __call__(self, features):
        labels = [feature.pop("labels") for feature in features] if "labels" in features[0] else None
        num_choices = len(features[0]["input_ids"])
        flattened = [
            {key: value[i] for key, value in feature.items()}
            for feature in features
            for i in range(num_choices)
        ]
        batch = self.tokenizer.pad(flattened, padding=True, return_tensors="pt")
        batch = {key: value.view(len(features), num_choices, -1) for key, value in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

collator = MultipleChoiceCollator(tokenizer)

def make_dataset(frame, include_labels):
    columns = ["prompt", *OPTION_LABELS] + (["labels"] if include_labels else [])
    ds = Dataset.from_pandas(frame[columns], preserve_index=False)
    return ds.map(
        lambda batch: tokenize_mcq(batch, with_labels=include_labels),
        batched=True,
        remove_columns=ds.column_names,
        desc="Tokenizing MCQs",
    )

train_indices, valid_indices = train_test_split(
    np.arange(len(mcq_train)),
    test_size=CFG["validation_size"],
    random_state=CFG["seed"],
    stratify=mcq_train["labels"],
)
train_ds = make_dataset(mcq_train.iloc[train_indices].reset_index(drop=True), include_labels=True)
valid_ds = make_dataset(mcq_train.iloc[valid_indices].reset_index(drop=True), include_labels=True)
test_ds = make_dataset(mcq_test.reset_index(drop=True), include_labels=False)
print(f"Split: {len(train_ds):,} train / {len(valid_ds):,} validation")

# %%
def make_model():
    model = AutoModelForMultipleChoice.from_pretrained(CFG["model_name"])
    model.config.use_cache = False
    # DeBERTa-v3 uses query_proj/key_proj/value_proj in its self-attention.
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=CFG["lora_r"],
        lora_alpha=CFG["lora_alpha"],
        lora_dropout=CFG["lora_dropout"],
        target_modules=["query_proj", "value_proj"],
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model

def map3_from_logits(logits, labels):
    ranks = np.argsort(-logits, axis=1)
    reciprocal_rank = np.array([
        1.0 / (np.where(rank == truth)[0][0] + 1)
        if truth in rank[:3] else 0.0
        for rank, truth in zip(ranks, labels)
    ])
    return float(reciprocal_rank.mean())

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    prediction_ids = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, prediction_ids),
        "macro_f1": f1_score(labels, prediction_ids, average="macro", zero_division=0),
        "map_at_3": map3_from_logits(logits, labels),
    }

# %%
# Do not set WANDB_MODE=disabled: the required Kaggle run is created here.
# In Kaggle, add WANDB_API_KEY as a Notebook secret before running this cell.
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

wandb_run = wandb.init(
    project="23f2000775-t22026",
    name=f"deberta-v3-lora-r{CFG['lora_r']}",
    config={**CFG, "model_type": "AutoModelForMultipleChoice", "choices": OPTION_LABELS},
    reinit=True,
)

wandb.define_metric("train/global_step")
wandb.define_metric("train/*", step_metric="train/global_step")
wandb.define_metric("eval/*", step_metric="train/global_step")

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta_mcq_lora",
    learning_rate=CFG["learning_rate"],
    per_device_train_batch_size=CFG["train_batch_size"],
    per_device_eval_batch_size=CFG["eval_batch_size"],
    gradient_accumulation_steps=CFG["gradient_accumulation"],
    num_train_epochs=CFG["epochs"],
    weight_decay=CFG["weight_decay"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    fp16=True,
    tf32=False,
    gradient_checkpointing=True,
    label_smoothing_factor=CFG["label_smoothing"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="map_at_3",
    greater_is_better=True,
    report_to=["wandb"],
    run_name=wandb_run.name,
    seed=CFG["seed"],
    dataloader_num_workers=2,
    optim="adamw_torch_fused",
)

model = make_model()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)],
)
trainer.train()
validation_metrics = trainer.evaluate(valid_ds, metric_key_prefix="validation")
print({k: round(v, 5) for k, v in validation_metrics.items() if isinstance(v, float)})
wandb.log({
    "validation/MAP@3": validation_metrics["validation_map_at_3"],
    "validation/accuracy": validation_metrics["validation_accuracy"],
    "validation/macro_f1": validation_metrics["validation_macro_f1"],
})
wandb.log({"validation/best_checkpoint": trainer.state.best_model_checkpoint or "final"})

# %%
# Train the *same* selected architecture on all labelled rows before producing
# the leaderboard file.  Validation remains logged above; this recovers the
# held-out examples for the final single-model submission.
if CFG["retrain_on_full_train"]:
    best_epoch = max(1, int(round(float(trainer.state.best_metric and trainer.state.epoch or CFG["epochs"]))))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    full_train_ds = make_dataset(mcq_train.reset_index(drop=True), include_labels=True)
    final_args = TrainingArguments(
        output_dir="/kaggle/working/deberta_mcq_lora_final",
        learning_rate=CFG["learning_rate"],
        per_device_train_batch_size=CFG["train_batch_size"],
        gradient_accumulation_steps=CFG["gradient_accumulation"],
        num_train_epochs=best_epoch,
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        lr_scheduler_type="cosine",
        fp16=True, tf32=False, gradient_checkpointing=True,
        label_smoothing_factor=CFG["label_smoothing"],
        logging_strategy="steps", logging_steps=20,
        save_strategy="no", report_to=["wandb"], run_name=wandb_run.name,
        seed=CFG["seed"], dataloader_num_workers=2, optim="adamw_torch_fused",
    )
    model = make_model()
    trainer = Trainer(
        model=model, args=final_args, train_dataset=full_train_ds,
        data_collator=collator, tokenizer=tokenizer,
    )
    trainer.train()

# %%
# Inference: logits -> softmax -> ranked top-three option labels, e.g. "A C B".
prediction_output = trainer.predict(test_ds)
test_logits = prediction_output.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
test_probabilities = F.softmax(torch.tensor(test_logits), dim=1).numpy()
top3_indices = np.argsort(-test_probabilities, axis=1)[:, :3]
top3_predictions = [" ".join(OPTION_LABELS[i] for i in row) for row in top3_indices]

if sample_paths:
    sample_submission = pd.read_csv(sample_paths[0])
    submission = sample_submission.copy()
    prediction_columns = [c for c in submission.columns if c != "id"]
    assert len(prediction_columns) == 1, f"Unexpected sample-submission columns: {submission.columns.tolist()}"
    submission[prediction_columns[0]] = top3_predictions
else:
    submission = pd.DataFrame({"id": mcq_test["id"], "prediction": top3_predictions})

assert len(submission) == len(mcq_test)
assert submission.iloc[:, 1].str.fullmatch(r"[A-E]( [A-E]){2}").all()
submission.to_csv("/kaggle/working/submission.csv", index=False)
display(submission.head())
print("Saved /kaggle/working/submission.csv", submission.shape)

# Persist the adapter and tokenizer as a Kaggle output artifact.
trainer.save_model("/kaggle/working/deberta_mcq_lora_adapter")
tokenizer.save_pretrained("/kaggle/working/deberta_mcq_lora_adapter")
wandb_run.finish()


Tesla T4
GPU count: 2
Train/test: (2000, 9) (500, 7) choices: ['A', 'B', 'C', 'D', 'E']


,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Tokenizing MCQs:   0%|          | 0/1760 [00:00<?, ? examples/s]

Tokenizing MCQs:   0%|          | 0/240 [00:00<?, ? examples/s]

...

trainable params: 590,593 || all params: 185,013,506 || trainable%: 0.3192


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,1.613200
40,1.609100
60,1.610400
80,1.610900
100,1.608800
120,1.613500
140,1.617200
160,1.611500
180,1.611300
200,1.612400


You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fas

AssertionError: Unexpected sample-submission columns: ['ID', 'Prediction']

In [18]:
import transformers
print(transformers.__version__)

4.57.6


In [25]:
print(submission.columns.tolist())

['ID', 'Prediction']


In [26]:
import numpy as np
import pandas as pd

# Predict on test set
prediction_output = trainer.predict(test_ds)
logits = prediction_output.predictions

# Top-3 predictions
top3_idx = np.argsort(logits, axis=1)[:, -3:][:, ::-1]
top3_predictions = [
    " ".join(OPTION_LABELS[i] for i in row)
    for row in top3_idx
]

# Build submission
if sample_paths:
    submission = pd.read_csv(sample_paths[0])

    # Automatically detect the prediction column
    pred_col = [c for c in submission.columns if c.lower() not in ["id"]][0]

    submission[pred_col] = top3_predictions
else:
    submission = pd.DataFrame({
        "ID": mcq_test["ID"] if "ID" in mcq_test.columns else mcq_test["id"],
        "Prediction": top3_predictions
    })

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("\nSaved to /kaggle/working/submission.csv")

You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


   ID Prediction
0   1      D E C
1   2      C A B
2   3      C A E
3   4      A B E
4   5      A C B

Saved to /kaggle/working/submission.csv
